In [1]:
# ==========================================
# 🚀 PROZORRO MONITOR v32.2: FIXED SETUP
# ==========================================
# 1. Исправлена ошибка 'not enough values to unpack'.
# 2. Логика "Якорь" + "Умный стоп" (остановка при дублях).
# 3. Полное логирование.

!pip install gspread oauth2client requests pytz -q

import time
import json
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime
import pytz
from kaggle_secrets import UserSecretsClient
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- НАСТРОЙКИ ---
MAX_RUNTIME = 11.8 * 60 * 60 
API_URL = "https://public.api.openprocurement.org/api/2.5/tenders"
THREADS = 20 
TZ_UA = pytz.timezone('Europe/Kiev')

# Критерии остановки
MIN_HISTORY_ANCHOR = 10  # Сколько тендеров собрать для базы при первом запуске
STOP_ON_DUPLICATES = 5   # Сколько дублей подряд найти, чтобы остановиться в режиме мониторинга

def get_google_client():
    user_secrets = UserSecretsClient()
    key_json = user_secrets.get_secret("GCP_KEY")
    sheet_url = user_secrets.get_secret("SHEET_URL")
    creds_dict = json.loads(key_json)
    scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
    client = gspread.authorize(creds)
    return client, sheet_url

def setup_sheets(sh):
    # 1. DATA
    try: ws_data = sh.worksheet("Data")
    except: 
        ws_data = sh.add_worksheet(title="Data", rows=1000, cols=10)
        print("⚙️ Лист Data создан.")
    
    headers = ['Дата', 'ID', 'Статус', 'Заказчик', 'Бюджет', 'CPV', 'Название', 'Дедлайн', 'Ссылка', 'Email Status']
    if not ws_data.row_values(1): ws_data.append_row(headers)

    # 2. SETTINGS
    try: ws_settings = sh.worksheet("Settings")
    except: 
        ws_settings = sh.add_worksheet(title="Settings", rows=20, cols=5)
        print("⚙️ Лист Settings создан.")
    
    if ws_settings.acell('A1').value != 'Параметр':
        ws_settings.update(range_name='A1:B4', values=[
            ['Параметр', 'Значение'], 
            ['CPV Codes (B2)', ''],            
            ['Search Status (B3)', 'active.tendering'],        
            ['Email (B4)', '']     
        ])
        
    # 3. LOGS (Исправлено: добавляем создание листа Logs)
    try: ws_logs = sh.worksheet("Logs")
    except:
        ws_logs = sh.add_worksheet(title="Logs", rows=1000, cols=4)
        ws_logs.append_row(['Timestamp', 'Event', 'Count', 'Details'])
        print("⚙️ Лист Logs создан.")

    return ws_data, ws_settings, ws_logs

def write_log(ws_logs, event, count=0, details=""):
    try:
        now = datetime.now(TZ_UA).strftime("%Y-%m-%d %H:%M:%S")
        ws_logs.insert_row([now, event, count, details], 2)
    except: pass

def read_settings(ws_settings):
    try:
        vals = ws_settings.get("B2:B4")
        raw_cpv = vals[0][0] if len(vals) > 0 and vals[0] else ""
        raw_status = vals[1][0] if len(vals) > 1 and vals[1] else ""
        raw_email = vals[2][0] if len(vals) > 2 and vals[2] else ""

        cpv_list = []
        if raw_cpv:
            parts = [c.strip() for c in raw_cpv.split(',')]
            for p in parts:
                clean = p.split('-')[0].strip()
                if clean:
                    short = clean.rstrip('0')
                    if len(short) < 2: short = clean[:2]
                    cpv_list.append(short)

        settings = {
            'cpv': cpv_list,
            'status': raw_status.strip(),
            'email': raw_email.strip()
        }
        print(f"📋 НАСТРОЙКИ: CPV={settings['cpv'] or 'ВСЕ'}, Статус='{settings['status']}'")
        return settings
    except Exception as e:
        print(f"⚠️ Ошибка настроек: {e}")
        return {'cpv': [], 'status': '', 'email': ''}

def get_existing_ids(ws_data):
    try:
        ids = ws_data.col_values(2)[1:]
        return set(ids)
    except: return set()

def fetch_full_tender(tid):
    try:
        r = requests.get(f"{API_URL}/{tid}", timeout=15)
        if r.status_code == 200: return r.json().get('data')
        elif r.status_code == 429: time.sleep(2); return None
    except: return None

def get_all_cpvs(t):
    codes = set()
    if t.get('classification'): 
        codes.add(t['classification'].get('id'))
    if t.get('items'):
        for item in t['items']:
            if item.get('classification'): 
                codes.add(item['classification'].get('id'))
    return codes

def check_tender_verbose(t, settings):
    t_status = t.get('status', 'unknown')
    if settings['status'] and t_status != settings['status']: 
        return False, 'status'
    
    found_codes = get_all_cpvs(t)
    if settings['cpv']:
        match = False
        for tc in found_codes:
            for prefix in settings['cpv']:
                if tc.startswith(prefix): 
                    match = True; break
            if match: break
        if not match: return False, 'cpv'
        
    return True, 'ok'

def process_tenders(candidates, settings, existing_ids, is_history_mode):
    full_tenders = []
    
    if candidates:
        with ThreadPoolExecutor(max_workers=THREADS) as executor:
            futures = {executor.submit(fetch_full_tender, i['id']): i['id'] for i in candidates}
            for future in as_completed(futures):
                res = future.result()
                if res: full_tenders.append(res)
    
    valid_rows = []
    stats = {'ok': 0, 'bad': 0}
    
    for t in full_tenders:
        is_ok, reason = check_tender_verbose(t, settings)
        
        if is_ok:
            if t['id'] in existing_ids: continue

            stats['ok'] += 1
            dt_str = datetime.fromisoformat(t['dateModified']).strftime("%Y-%m-%d %H:%M")
            dl = "-"
            if t.get('tenderPeriod') and t['tenderPeriod'].get('endDate'):
                    try: dl = datetime.fromisoformat(t['tenderPeriod']['endDate']).strftime("%Y-%m-%d %H:%M")
                    except: pass
            
            all_codes = get_all_cpvs(t)
            cpv_str = ", ".join(all_codes) if all_codes else "-"
            
            email_status = "History" if is_history_mode else "Pending"
            
            row = [
                dt_str, t['id'], t.get('status', ''),
                t.get('procuringEntity', {}).get('name', '-'),
                t.get('value', {}).get('amount', 0),
                cpv_str, t.get('title', '')[:250], dl, 
                f"https://prozorro.gov.ua/tender/{t['id']}",
                email_status
            ]
            valid_rows.append(row)
        else:
            stats['bad'] += 1
            
    return valid_rows, stats

def main_loop():
    client, sheet_url = get_google_client()
    sh = client.open_by_url(sheet_url)
    ws_data, ws_settings, ws_logs = setup_sheets(sh)
    
    print("\n🚀 ЗАПУСК МОНИТОРА (v32.2 - FIXED)...")
    write_log(ws_logs, "Bot Started", 0, "Start")

    start_time = time.time()
    
    while True:
        if (time.time() - start_time) > MAX_RUNTIME:
            print("🛑 Смена окончена.")
            write_log(ws_logs, "Bot Stopped", 0, "Timeout")
            break
        
        settings = read_settings(ws_settings)
        existing_ids = get_existing_ids(ws_data)
        
        # Определяем фазу
        is_history_mode = len(existing_ids) < MIN_HISTORY_ANCHOR
        mode_name = "HISTORY" if is_history_mode else "MONITOR"
        
        # Порог остановки
        current_stop_threshold = min(STOP_ON_DUPLICATES, len(existing_ids)) if len(existing_ids) > 0 else 999999
        
        print(f"\n🔄 Режим: {mode_name}. База: {len(existing_ids)}. Стоп после {current_stop_threshold} дублей.")
        
        url = f"{API_URL}?descending=1&limit=50&opt_fields=id,status,dateModified"
        
        new_rows_buffer = []
        found_in_cycle = 0
        duplicates_streak = 0
        stop_scan = False
        
        while not stop_scan:
            try:
                r = requests.get(url, timeout=30)
                if r.status_code == 429:
                    print("⏳ Лимит API. Жду 60 сек...")
                    time.sleep(60); continue
                
                data = r.json()
                items = data.get('data', [])
                
                if not items:
                    print("✅ Лента закончилась.")
                    break
                
                last_dt = items[0]['dateModified']
                print(f"   📥 Пачка {len(items)} шт. | {last_dt}")
                
                candidates = []
                
                for t in items:
                    if t['id'] in existing_ids:
                        duplicates_streak += 1
                        # В режиме мониторинга считаем дубли подряд
                        if not is_history_mode:
                            if duplicates_streak >= current_stop_threshold:
                                print(f"   🛑 Набрано {duplicates_streak} дублей. Дошли до прочитанного.")
                                stop_scan = True
                                break 
                        continue
                    else:
                        # Если нашли новый - добавляем в кандидаты
                        # Внимание: streak сбрасываем только если это реально подходящий тендер?
                        # Нет, если ID новый - значит это разрыв в дублях.
                        duplicates_streak = 0
                        candidates.append(t)
                
                if stop_scan and not candidates: break
                
                if candidates:
                    valid_rows, stats = process_tenders(candidates, settings, existing_ids, is_history_mode)
                    print(f"      + {stats['ok']} (Отсев: {stats['bad']})")
                    
                    if valid_rows:
                        try:
                            ws_data.insert_rows(valid_rows, 2)
                            print(f"💾 СОХРАНЕНО: {len(valid_rows)} строк.")
                            write_log(ws_logs, f"Data Saved ({mode_name})", len(valid_rows), f"Last: {valid_rows[0][1]}")

                            for row in valid_rows: existing_ids.add(row[1])
                            found_in_cycle += len(valid_rows)
                        except Exception as e:
                            print(f"🔥 Ошибка записи: {e}")
                            write_log(ws_logs, "Write Error", 0, str(e))
                            stop_scan = True

                if is_history_mode and len(existing_ids) >= MIN_HISTORY_ANCHOR:
                    print(f"   🎉 База наполнена ({len(existing_ids)} шт).")
                    stop_scan = True
                
                if stop_scan: break
                if 'next_page' in data: url = data['next_page']['uri']
                else: break

            except Exception as e:
                print(f"❌ Ошибка: {e}")
                time.sleep(30)
                
        print("⏳ Жду 5 минут...")
        time.sleep(300)

if __name__ == "__main__":
    main_loop()


🚀 ЗАПУСК МОНИТОРА (v32.2 - FIXED)...
📋 НАСТРОЙКИ: CPV=['181', '182', '184', '188', '3511', '3581'], Статус='active.tendering'

🔄 Режим: MONITOR. База: 4721. Стоп после 5 дублей.
   📥 Пачка 50 шт. | 2026-10-12T19:53:46.810966+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T19:46:56.336008+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T19:37:09.936865+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T19:28:19.912717+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T19:20:01.501117+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T19:09:19.782638+03:00
      + 0 (Отсев: 49)
   📥 Пачка 50 шт. | 2026-10-12T19:00:18.930286+03:00
      + 5 (Отсев: 45)
💾 СОХРАНЕНО: 5 строк.
   📥 Пачка 50 шт. | 2026-10-12T18:53:22.751513+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T18:44:35.987039+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T18:38:33.157926+03:00
      + 0 (Отсев: 50)
   📥 Пачка 50 шт. | 2026-10-12T18:32:31.976020+03